# Unit III assignment: Train, validate, test
**CS 301 · Learning algorithms**

Build a small neural-network classifier with scikit-learn. Your main job is to repeat a disciplined learning loop: **fit on training data → compare on validation data → freeze your choices → evaluate on test data**.

You will choose **one** dataset, compare **4–6** model configurations, and repeat the two strongest candidates with **five initializations each**. Aim for the best validation performance you can justify within that budget. A larger model or a lucky run need not be better.

## How to work
- In Colab, choose **File → Save a copy in Drive**. A CPU runtime is sufficient.
- Run the cells in order. **Your turn** marks short code edits and written responses; replace each `...` before continuing.
- Write a prediction before an experiment, then explain the actual result. Improvement is not required: an unsuccessful experiment can be informative.
- Keep the dataset and split fixed. Leave the test set alone until the final evaluation.
- Submit your completed `.ipynb` with outputs and explanations, using the assignment instructions and deadline on Canvas. A high accuracy by itself is not the goal; the procedure and your interpretation matter.

**Name:** *Your name here.*


In [ ]:
import copy
import warnings
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from IPython.display import display
from sklearn.datasets import load_digits, load_wine, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, log_loss, ConfusionMatrixDisplay
from sklearn.exceptions import ConvergenceWarning
from threadpoolctl import threadpool_limits

SPLIT_SEED = 301          # Which observations go into each set.
TRIAL_SEED = 0           # Same initialization seed for the first comparison.
INITIALIZATION_SEEDS = [0, 1, 2, 3, 4]
FINAL_SEED = 0           # Fixed now; do not search for a lucky final seed.
MAX_EPOCHS = 300
TEST_OPENED = False
print('Python:', platform.python_version(), '| scikit-learn:', sklearn.__version__)


## 1. Choose one prediction task

All three datasets are included with scikit-learn. They have numeric inputs and class labels, so the rest of the notebook is the same for each choice.

| Choice | One observation | Predict | Rows / features / classes |
|---|---|---|---|
| `digits` (default) | An 8 × 8 image represented by 64 pixel values | Which digit, 0–9 | 1,797 / 64 / 10 |
| `wine` | 13 chemical measurements of a wine sample | Which of three cultivars it belongs to, **not its quality** | 178 / 13 / 3 |
| `breast_cancer` | 30 measurements derived from a cell-nuclei image | The recorded benign/malignant label | 569 / 30 / 2 |

Choose for your interest in the task. Scores across these datasets are not directly comparable. These small historical datasets let us practice the learning procedure; their results do not establish performance on a new population.

**Your turn:** choose one dataset below and keep it for the entire assignment.


In [ ]:
DATASET = 'digits'  # Choose 'digits', 'wine', or 'breast_cancer'.
loaders = {'digits': load_digits, 'wine': load_wine, 'breast_cancer': load_breast_cancer}
assert DATASET in loaders, 'Choose one of the three listed dataset names.'
data = loaders[DATASET]()
X, y = data.data, data.target
class_names = np.asarray(data.target_names).astype(str)
print(DATASET, '| rows:', len(y), '| features:', X.shape[1], '| classes:', len(class_names))


## 2. Split before fitting anything

| Set | Approximate share | Its job |
|---|---:|---|
| Training | 60% | Fit the scaler and the network's weights and biases. |
| Validation | 20% | Compare configurations and examine initialization effects. |
| Test | 20% | Evaluate the chosen procedure once, after choices are fixed. |

The supplied code first reserves 20% for testing, then takes 25% of the remaining 80% for validation: **0.25 × 0.80 = 0.20**. Stratification keeps class proportions approximately similar. Exact sizes depend on rounding.

The split seed controls the data partition. A model seed controls the network's random initialization. Changing one is a different experiment from changing the other.


In [ ]:
row_ids = np.arange(len(y))
development_ids, test_ids = train_test_split(
    row_ids, test_size=0.20, stratify=y, random_state=SPLIT_SEED)
train_ids, valid_ids = train_test_split(
    development_ids, test_size=0.25, stratify=y[development_ids], random_state=SPLIT_SEED)

X_train, y_train = X[train_ids], y[train_ids]
X_valid, y_valid = X[valid_ids], y[valid_ids]
X_test, y_test = X[test_ids], y[test_ids]  # Reserved until Section 7.

assert set(train_ids).isdisjoint(valid_ids)
assert set(train_ids).isdisjoint(test_ids)
assert set(valid_ids).isdisjoint(test_ids)
assert len(train_ids) + len(valid_ids) + len(test_ids) == len(y)
display(pd.DataFrame({'set': ['training', 'validation', 'test'],
                      'rows': [len(train_ids), len(valid_ids), len(test_ids)]}))
print('Checks passed: every row belongs to exactly one set.')


In [ ]:
# Inspect training observations, not test outcomes.
display(pd.DataFrame(X_train[:5], columns=data.feature_names))
display(pd.DataFrame({'class': class_names,
                      'training_count': np.bincount(y_train, minlength=len(class_names))}))
if DATASET == 'digits':
    fig, axes = plt.subplots(1, 8, figsize=(10, 2))
    for ax, image, label in zip(axes, X_train[:8], y_train[:8]):
        ax.imshow(image.reshape(8, 8), cmap='gray_r')
        ax.set_title(str(label)); ax.axis('off')
    plt.show()


**Your turn — describe and explain.**
1. In your own words, what is one row, what are its inputs, and what does its target mean?
2. Record your three split sizes. What is the role of each set?
3. Why must we split before learning the mean and scale of each feature?

*Write your response here.*


## 3. Establish a baseline, then fit the first MLP

A majority-class classifier always predicts the most common **training** label. It gives us a reference that requires no neural network. Predict its validation accuracy approximately from the training class counts before running the next cell.

**Your prediction:** *Write it here.*


In [ ]:
baseline = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
baseline_valid_accuracy = accuracy_score(y_valid, baseline.predict(X_valid))
print(f'Majority-class validation accuracy: {baseline_valid_accuracy:.3f}')


An MLP is a network with one or more hidden layers. We use `StandardScaler` followed by `MLPClassifier` in a **pipeline**. Calling `fit` learns the feature scaling and the network from the same training rows; `predict` applies that learned scaling to new rows.

| Setting you choose | Meaning |
|---|---|
| `hidden_layer_sizes=(16,)` | One hidden layer with 16 neurons. `(32, 16)` means two hidden layers. |
| `alpha=0.0001` | Strength of the penalty on large weights. Larger values increase this regularization. |
| `learning_rate_init=0.001` | Initial learning rate used by Adam. |

The network's weights and biases are **learned parameters**. The settings above are **hyperparameters** that you compare using validation. ReLU and Adam remain fixed in this assignment.

`max_iter=300` is a maximum of 300 **epochs**, not 300 individual gradient updates. A run can stop earlier when training loss stops improving. We record the actual epoch count. A `budget_warning=True` entry means the run reached the cap before its stopping criterion was met; it is a usable finite-budget result, not evidence of convergence. Keep these runs in your table.

**Your turn:** complete the three `...` expressions in `run_trial`. Fit on training inputs and targets; predict the validation inputs. The function will repeat that procedure for every trial. It never uses the test set.


In [ ]:
def make_model(config, seed):
    return Pipeline([
        ('scale', StandardScaler()),
        ('mlp', MLPClassifier(
            hidden_layer_sizes=config['hidden_layer_sizes'],
            alpha=config['alpha'], learning_rate_init=config['learning_rate_init'],
            activation='relu', solver='adam', max_iter=MAX_EPOCHS,
            early_stopping=False, shuffle=False, random_state=seed))
    ])

def run_trial(config, seed):
    assert not TEST_OPENED, 'Model selection is finished once the test set is opened.'
    model = make_model(config, seed)
    fit_inputs = ...   # TODO: inputs used to learn scaling and network parameters.
    fit_targets = ...  # TODO: corresponding targets.
    validation_inputs = ...  # TODO: inputs used to compare model choices.
    assert all(v is not Ellipsis for v in [fit_inputs, fit_targets, validation_inputs]), 'Complete the three expressions first.'
    with threadpool_limits(limits=1), warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always', ConvergenceWarning)
        model.fit(fit_inputs, fit_targets)
    assert np.allclose(model.named_steps['scale'].mean_, X_train.mean(axis=0)), 'The scaler must be fitted on training rows only.'
    valid_prediction = model.predict(validation_inputs)
    mlp = model.named_steps['mlp']
    row = dict(name=config['name'], seed=seed,
               train_accuracy=accuracy_score(y_train, model.predict(X_train)),
               valid_accuracy=accuracy_score(y_valid, valid_prediction),
               valid_log_loss=log_loss(y_valid, model.predict_proba(validation_inputs)),
               epochs=mlp.n_iter_,
               budget_warning=any(issubclass(w.category, ConvergenceWarning) for w in caught))
    return model, row


In [ ]:
STARTER = dict(name='small', hidden_layer_sizes=(16,), alpha=0.0001, learning_rate_init=0.001)
first_model, first_result = run_trial(STARTER, TRIAL_SEED)
display(pd.DataFrame([first_result]).round(4))
plt.plot(first_model.named_steps['mlp'].loss_curve_)
plt.xlabel('Epoch'); plt.ylabel('Training objective'); plt.title('First MLP: progress during fitting')
plt.grid(alpha=0.25); plt.show()


**Your turn — read the result.** Compare training accuracy, validation accuracy, and the majority-class baseline. What does the loss curve tell you about training? Why does a falling training objective alone not establish good predictions on new observations?

The plotted training objective includes the classification loss and the weight penalty. Accuracy is the fraction of correct labels (**higher is better**). Validation log loss evaluates the predicted probabilities (**lower is better**). Two models can have the same accuracy but different log loss.

*Write your response using your numbers.*


## 4. Repeat the loop: change, fit, validate, record

Create **four distinct configurations in total**, including the starter. Two are supplied; add two of your own. You may add at most two further trials if you have a specific question, for a total of **4–6**.

- Vary at least **two kinds** of hyperparameter across the table.
- Change one setting at a time when making a comparison you want to explain.
- Suggested choices: hidden layers `(16,)`, `(32,)`, `(64,)`, or `(32, 16)`; `alpha` in `0.0001, 0.01, 0.1`; learning rate in `0.001, 0.003, 0.01`.
- Keep all trials, including unsuccessful ones. Keep the split, initial seed, and maximum epoch budget fixed.
- Aim for the highest **validation accuracy**. Use lower validation log loss to break an accuracy tie. Do not choose a winner using training accuracy or test results.

Repeated validation comparisons can themselves favor chance successes. Our small trial budget and the later initialization check help make the evidence easier to interpret; the untouched test set still has a separate role.

**Predict before running:** name the changes you will try and why each might help or hurt. A reasoned prediction can be wrong.

| Trial name | Change from which earlier trial? | Prediction and reason |
|---|---|---|
| wider | small: 16 → 32 hidden neurons | *Your prediction* |
| *your trial* | *Your change* | *Your prediction* |
| *your trial* | *Your change* | *Your prediction* |


In [ ]:
TRIALS = [
    STARTER,
    dict(name='wider', hidden_layer_sizes=(32,), alpha=0.0001, learning_rate_init=0.001),
    ...,  # TODO: a dictionary with a unique name and the same four keys.
    ...,  # TODO: a second configuration of your own.
]
assert 4 <= len(TRIALS) <= 6
assert all(isinstance(c, dict) for c in TRIALS), 'Replace each ... with a configuration dictionary.'
assert len({c['name'] for c in TRIALS}) == len(TRIALS), 'Use a distinct name for each trial.'
keys = ('hidden_layer_sizes', 'alpha', 'learning_rate_init')
signatures = [tuple(c[k] for k in keys) for c in TRIALS]
assert len(set(signatures)) == len(TRIALS), 'Use distinct configurations.'
assert sum(len({c[k] for c in TRIALS}) > 1 for k in keys) >= 2, 'Vary at least two kinds of hyperparameter.'


In [ ]:
models, rows = {}, []
for config in TRIALS:
    model, row = run_trial(config, TRIAL_SEED)
    models[config['name']] = model
    rows.append({**row, **{k: config[k] for k in keys}})
results = pd.DataFrame(rows)
display(results.round(4))
print('The test set has not been scored.')


**Your turn — explain two comparisons.**
1. Which single-setting change helped validation performance most, or hurt it least? Cite both trial names and numbers.
2. Discuss a change that did not help as you expected. If all changes behaved as predicted, discuss the least useful change or a remaining uncertainty. Use the training/validation gap and any budget warning as evidence.
3. Which two configurations currently look strongest under our accuracy-then-log-loss rule?

*Write your response here. If several trials tie, say so and use their log losses.*


## 5. Was the result dependent on initialization?

The next experiment repeats the two strongest configurations using seeds **0, 1, 2, 3, 4**. Keep the data partition, scaling procedure, architecture, learning rate, regularization, and training budget fixed within each configuration.

`shuffle=False` fixes the order of training batches. `early_stopping=False` avoids an additional random internal validation split. Thus changing the model seed changes initialization in this controlled experiment. With the usual shuffled training, a seed can also change the batch order.

We compare **mean validation accuracy across the same five seeds**, then mean validation log loss for a tie. Do not choose the single luckiest seed. This is five separately fitted models, not an ensemble.

**Predict before running:** will the two candidates retain their ordering? Must every seed produce a different accuracy?

*Write your prediction here.*


In [ ]:
ranked = results.sort_values(['valid_accuracy', 'valid_log_loss', 'name'], ascending=[False, True, True])
candidate_names = ranked.head(2)['name'].tolist()
config_by_name = {c['name']: c for c in TRIALS}
seed_rows = []
for name in candidate_names:
    for seed in INITIALIZATION_SEEDS:
        _, row = run_trial(config_by_name[name], seed)
        seed_rows.append(row)
seed_results = pd.DataFrame(seed_rows)
summary = seed_results.groupby('name', sort=False).agg(
    mean_valid_accuracy=('valid_accuracy', 'mean'),
    sd_valid_accuracy=('valid_accuracy', 'std'),
    min_valid_accuracy=('valid_accuracy', 'min'),
    max_valid_accuracy=('valid_accuracy', 'max'),
    mean_valid_log_loss=('valid_log_loss', 'mean'),
    budget_warnings=('budget_warning', 'sum')).reset_index()
summary = summary.sort_values(['mean_valid_accuracy', 'mean_valid_log_loss', 'name'], ascending=[False, True, True])
display(seed_results.round(4))
display(summary.round(4))
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for name in candidate_names:
    r = seed_results[seed_results['name'] == name]
    axes[0].plot(r['seed'], r['valid_accuracy'], 'o-', label=name)
    axes[1].plot(r['seed'], r['valid_log_loss'], 'o-', label=name)
for ax, label in zip(axes, ['Validation accuracy (higher is better)', 'Validation log loss (lower is better)']):
    ax.set_xlabel('Initialization seed'); ax.set_ylabel(label)
    ax.set_xticks(INITIALIZATION_SEEDS); ax.grid(alpha=0.25); ax.legend()
plt.tight_layout(); plt.show()


**Your turn — explain the variation.** Report each candidate's mean and range of validation accuracy. Did the original winner remain the winner? What would be misleading about reporting only the best seed?

If accuracies tie, inspect log loss: identical numbers of correct labels do not imply identical probabilities or learned weights. Do not change the split or keep trying seeds to manufacture a larger difference. The standard deviation describes these five initializations on this split; it is not a confidence interval over new populations.

*Write your response using the tables and plots.*


## 6. Freeze the choice before testing

Choose the configuration with the highest mean validation accuracy in the five-seed comparison. Break a tie using lower mean validation log loss. If both tie, either is acceptable with an explanation.

The final initialization seed was fixed at **0** in the setup. We select a configuration, not a seed. The final model will be fitted on the **training rows only**, using a new pipeline with that same predeclared seed. The validation and test rows remain outside fitting throughout this assignment.

**Your turn:** enter the chosen trial name below. Then explain your selection before running Section 7.


In [ ]:
FINAL_NAME = ...  # TODO: a name from the five-seed summary, as a string.
assert FINAL_NAME in candidate_names, 'Enter one of the two candidate names.'
chosen_row = summary.set_index('name').loc[FINAL_NAME]
best = summary.iloc[0]
assert np.isclose(chosen_row['mean_valid_accuracy'], best['mean_valid_accuracy']), 'Use the highest mean validation accuracy.'
assert np.isclose(chosen_row['mean_valid_log_loss'], best['mean_valid_log_loss']), 'For an accuracy tie, use lower mean validation log loss.'
FINAL_CONFIG = copy.deepcopy(config_by_name[FINAL_NAME])
print('Frozen configuration:', FINAL_CONFIG)
print('Final seed:', FINAL_SEED, '| Test set still unopened:', not TEST_OPENED)


**Before opening the test set:** record your final configuration, its mean validation accuracy, accuracy range, and mean log loss. Explain why the evidence supports this choice. Why would choosing a new configuration after looking at test accuracy change the role of the test set?

*Write your response here before continuing.*


## 7. Evaluate once on the reserved test set

The next cell fits the frozen model on training data and reports its test performance alongside the predeclared majority baseline. The confusion matrix below uses the same final predictions; it is a description of errors, not another model-selection round.

After opening the test set, stop tuning. If the score disappoints, report and explain it. A best validation result does not guarantee a best test result. You may later restart and rerun the **same completed notebook** to reproduce it for submission, keeping all choices unchanged.


In [ ]:
assert not TEST_OPENED, 'This test evaluation has already been run in this session.'
final_model = make_model(FINAL_CONFIG, FINAL_SEED)
with threadpool_limits(limits=1), warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always', ConvergenceWarning)
    final_model.fit(X_train, y_train)
test_prediction = final_model.predict(X_test)
test_probability = final_model.predict_proba(X_test)
TEST_OPENED = True
test_accuracy = accuracy_score(y_test, test_prediction)
test_log_loss = log_loss(y_test, test_probability)
baseline_test_accuracy = accuracy_score(y_test, baseline.predict(X_test))
display(pd.DataFrame([
    {'model': 'Majority baseline', 'test_accuracy': baseline_test_accuracy, 'test_log_loss': np.nan},
    {'model': FINAL_NAME, 'test_accuracy': test_accuracy, 'test_log_loss': test_log_loss}
]).round(4))
print('Final fit epochs:', final_model.named_steps['mlp'].n_iter_)
print('Final fit budget warning:', any(issubclass(w.category, ConvergenceWarning) for w in caught))
print('Model selection is finished. Describe these results without further tuning.')


In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test, test_prediction, labels=np.arange(len(class_names)), display_labels=class_names,
    colorbar=False, cmap='Blues', ax=ax)
ax.set_title('Frozen model: test errors')
plt.tight_layout(); plt.show()


## 8. Explain what you learned

Write a short conclusion (about 150–200 words) using your actual results:
- State the task, final configuration, validation evidence, and final test accuracy.
- Describe one useful hyperparameter experiment and one result that surprised you.
- Explain what the initialization experiment changed in your interpretation of a single run.
- Identify one error pattern in the confusion matrix. Small class counts limit what you can conclude.
- Explain why your test score is an evaluation of the frozen procedure, and name one limitation of this experiment.

**Your conclusion:** *Write here.*


## Final check and submission

Your notebook should contain:
1. One dataset choice, its description, and the three disjoint split sizes.
2. The completed fit/predict code and training-only preprocessing.
3. Predictions and results for 4–6 distinct configurations, varying at least two hyperparameters.
4. Five initializations of each of two candidates, with the table, summary, and explanation.
5. A validation-based selection recorded before the final test evaluation.
6. Final test results, a confusion matrix, and your conclusion.
7. Credits and any required AI disclosure/appendix below.

Restart the runtime and run your **completed, unchanged choices** from top to bottom. Check that the figures and tables appear and your responses are filled in. Save with outputs, then **File → Download → Download .ipynb** and submit as instructed on Canvas. Do not submit only screenshots or a link that requires permission to view.


## Credits and AI disclosure

Follow the course syllabus: submitted work must reflect your own understanding, and external sources and collaborators must be credited. If you used an AI tool, explicitly disclose that use and include the **full interaction transcript as an appendix** in the submitted work. Verify any code you use.

**Sources or collaborators:** *List them here, or write “none.”*

**AI use:** *Describe it here, or write “none.” If used, add the full transcript below as an appendix.*


## Reference links

- [MLPClassifier: parameters and stopping behavior](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html)
- [Neural networks: scaling, hyperparameters, and initialization](https://scikit-learn.org/stable/modules/neural_networks_supervised.html)
- [Avoiding data leakage with preprocessing and pipelines](https://scikit-learn.org/stable/common_pitfalls.html)
- Dataset descriptions and provenance: [digits](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html), [wine](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_wine.html), [breast cancer](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html).
